## 0. Setup and how to use this notebook

Run each cell from top to bottom. Predict outputs first, then discuss *why* they appear. The five activities in this repository provide hands-on TODO exercises based on the same concepts.

In [1]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__, "| NumPy:", np.__version__)

pandas: 3.0.3 | NumPy: 2.4.6


## 1. pandas Series — labels and position (slides 12–19)

A **Series** is a one-dimensional sequence with an **index**. `.loc` selects by label, while `.iloc` selects by integer position.

In [2]:
scores = pd.Series(
    [36, 73, 56, 66, 87],
    index=["Mary", "Jack", "Ahmed", "June", "Hassan"],
    name="score",
)
print(scores)
print("Ahmed by label:", scores.loc["Ahmed"])
print("Third by position:", scores.iloc[2])
print("Mean:", scores.mean(), "| Maximum:", scores.max())
print("At least 60:\n", scores[scores >= 60])

Mary      36
Jack      73
Ahmed     56
June      66
Hassan    87
Name: score, dtype: int64
Ahmed by label: 56
Third by position: 56
Mean: 63.6 | Maximum: 87
At least 60:
 Jack      73
June      66
Hassan    87
Name: score, dtype: int64


### Series alignment — a frequent source of missing values

pandas aligns **index labels** during arithmetic, even if the two Series have different lengths or orders.

In [3]:
bonus = pd.Series({"Jack": 5, "Hassan": 5, "Noura": 10})
print("Ordinary addition (unmatched labels become NaN):\n", scores + bonus)
print("Addition with an explicit zero for absent labels:\n", scores.add(bonus, fill_value=0))

Ordinary addition (unmatched labels become NaN):
 Ahmed      NaN
Hassan    92.0
Jack      78.0
June       NaN
Mary       NaN
Noura      NaN
dtype: float64
Addition with an explicit zero for absent labels:
 Ahmed     56.0
Hassan    92.0
Jack      78.0
June      66.0
Mary      36.0
Noura     10.0
dtype: float64


## 2. DataFrames — create, inspect, select and rank (slides 24–41)

A **DataFrame** is a labelled two-dimensional table. Inspect shape, column types and missingness before changing data.

In [4]:
students = pd.DataFrame({
    "Name": ["Mary", "Jack", "Ahmed", "June", "Hassan"],
    "Score": [36, 73, 56, 66, 87],
})
print(students)
print("Shape:", students.shape)
print("Types:\n", students.dtypes)
print("One column is a Series:", type(students["Score"]).__name__)
print("Double brackets give a DataFrame:", type(students[["Score"]]).__name__)

     Name  Score
0    Mary     36
1    Jack     73
2   Ahmed     56
3    June     66
4  Hassan     87
Shape: (5, 2)
Types:
 Name       str
Score    int64
dtype: object
One column is a Series: Series
Double brackets give a DataFrame: DataFrame


In [5]:
students["Rank"] = students["Score"].rank(ascending=False, method="min").astype(int)
students["Passed"] = students["Score"] >= 60
print("Ranked results:\n", students.sort_values("Score", ascending=False).to_string(index=False))
print("\nOnly students scoring >=60:\n", students.loc[students["Passed"], ["Name", "Score"]])
print("\nSecond original row (.iloc):\n", students.iloc[1])

Ranked results:
   Name  Score  Rank  Passed
Hassan     87     1    True
  Jack     73     2    True
  June     66     3    True
 Ahmed     56     4   False
  Mary     36     5   False

Only students scoring >=60:
      Name  Score
1    Jack     73
3    June     66
4  Hassan     87

Second original row (.iloc):
 Name      Jack
Score       73
Rank         2
Passed    True
Name: 1, dtype: object


### Avoid chained assignment

When changing values in a subset, use `.loc[row_condition, "Column"] = value` or modify a deliberate `.copy()` of the subset. This makes the intended target explicit.

In [6]:
example = students.copy()
example.loc[example["Name"] == "Mary", "Score"] = 40
print("Updated copy:", example.loc[example["Name"] == "Mary", "Score"].iloc[0])
print("Original unchanged:", students.loc[students["Name"] == "Mary", "Score"].iloc[0])

Updated copy: 40
Original unchanged: 36


## 3. Missing values — detect, choose, validate (slides 46–66)

Missing values are often represented by `np.nan`. Always quantify gaps before deciding to drop, replace, interpolate or impute them.

In [7]:
measurements = pd.Series([23.0, np.nan, 31.0, 40.0, np.nan], name="measurement")
print("Original:\n", measurements)
print("Missing count:", measurements.isna().sum())
print("Missing percentage:", 100 * measurements.isna().mean())
print("Median fill:\n", measurements.fillna(measurements.median()))
print("Drop missing:\n", measurements.dropna())

Original:
 0    23.0
1     NaN
2    31.0
3    40.0
4     NaN
Name: measurement, dtype: float64
Missing count: 2
Missing percentage: 40.0
Median fill:
 0    23.0
1    31.0
2    31.0
3    40.0
4    31.0
Name: measurement, dtype: float64
Drop missing:
 0    23.0
2    31.0
3    40.0
Name: measurement, dtype: float64


In [8]:
rng = np.random.default_rng(42)
values = rng.integers(20, 100, size=20).astype(float)
values[[2, 5, 9, 13, 17]] = np.nan
s = pd.Series(values, name="measurement")
with_888 = s.fillna(888)
with_median = s.fillna(s.median())
print("Missing:", s.isna().sum(), "out of", len(s))
print("Mean with 888:", round(with_888.mean(), 2))
print("Mean with median:", round(with_median.mean(), 2))
print("Sentinel -999 repaired:", pd.Series([12.0, -999.0, 15.0]).replace(-999.0, np.nan).interpolate().tolist())

Missing: 5 out of 20
Mean with 888: 269.75
Mean with median: 63.25
Sentinel -999 repaired: [12.0, 13.5, 15.0]


## 4. Combining tables — concat, merge, join (slides 68–94)

- `pd.concat()` **stacks** compatible tables along rows or columns.
- `.merge()` **matches** records on shared key columns.
- `.join()` is convenient when a lookup is indexed by its key.

When enriching orders from a lookup, use a **left** join and validate key cardinality.

In [9]:
jan = pd.DataFrame({"OrderID": [1, 2], "CustomerID": [10, 11]})
feb = pd.DataFrame({"OrderID": [3, 4], "CustomerID": [10, 99]})
customers = pd.DataFrame({"CustomerID": [10, 11], "City": ["Leeds", "Berlin"]})
orders = pd.concat([jan, feb], ignore_index=True)
print("Concatenated orders:\n", orders)
merged = orders.merge(customers, on="CustomerID", how="left",
                      validate="many_to_one", indicator=True)
print("\nAudited left merge:\n", merged)
print("\nUnknown customer orders:\n", merged.loc[merged["_merge"] == "left_only"])
print("\nEquivalent index join:\n", orders.join(customers.set_index("CustomerID"), on="CustomerID"))

Concatenated orders:
    OrderID  CustomerID
0        1          10
1        2          11
2        3          10
3        4          99

Audited left merge:
    OrderID  CustomerID    City     _merge
0        1          10   Leeds       both
1        2          11  Berlin       both
2        3          10   Leeds       both
3        4          99     NaN  left_only

Unknown customer orders:
    OrderID  CustomerID City     _merge
3        4          99  NaN  left_only

Equivalent index join:
    OrderID  CustomerID    City
0        1          10   Leeds
1        2          11  Berlin
2        3          10   Leeds
3        4          99     NaN


### Understand join types and duplicate-key risk

An inner join retains only matched keys. A left join retains all keys from the left table. An outer join retains keys from either side. Repeated lookup keys can unexpectedly multiply rows.

In [10]:
left = pd.DataFrame({"ID": [1, 2], "L": ["A", "B"]})
right = pd.DataFrame({"ID": [2, 3], "R": ["X", "Y"]})
for how in ["inner", "left", "outer"]:
    result = left.merge(right, on="ID", how=how, indicator=True)
    print(f"\n{how.upper()} JOIN:\n{result.to_string(index=False)}")
try:
    duplicate_lookup = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)
    orders.merge(duplicate_lookup, on="CustomerID", validate="many_to_one")
except pd.errors.MergeError as exc:
    print("\nExpected validation failure:", exc)


INNER JOIN:
 ID L R _merge
  2 B X   both

LEFT JOIN:
 ID L   R    _merge
  1 A NaN left_only
  2 B   X      both

OUTER JOIN:
 ID   L   R     _merge
  1   A NaN  left_only
  2   B   X       both
  3 NaN   Y right_only

Expected validation failure: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  CustomerID
         10 ...


## 5. Integrated mini-case — reconcile retail revenue (slides 96–102)

Calculate line revenue after enriching orders with product category and customer city. A missing city should not make a valid order disappear.

In [11]:
retail_orders = pd.DataFrame({
    "OrderID": [1, 2, 3, 4, 5], "CustomerID": [10, 11, 10, 12, 99],
    "ProductID": ["P1", "P2", "P1", "P3", "P2"],
    "Quantity": [2, 1, 3, 1, 2], "UnitPrice": [5., 20., 5., 40., 20.],
})
products = pd.DataFrame({"ProductID": ["P1", "P2", "P3"],
                         "Category": ["Stationery", "Books", "Accessories"]})
customer_lookup = pd.DataFrame({"CustomerID": [10, 11, 12],
                                "City": ["Leeds", "Berlin", "Leeds"]})
report = retail_orders.merge(products, on="ProductID", how="left", validate="many_to_one")
report = report.merge(customer_lookup, on="CustomerID", how="left",
                      validate="many_to_one", indicator=True)
report["Revenue"] = report["Quantity"] * report["UnitPrice"]
print("Total:", report["Revenue"].sum())
print("Category revenue:\n", report.groupby("Category")["Revenue"].sum())
print("City revenue (unknown retained):\n", report.groupby("City", dropna=False)["Revenue"].sum())
assert len(report) == 5 and report["Revenue"].sum() == 125
assert report.groupby("City", dropna=False)["Revenue"].sum().sum() == 125
print("Validation checks passed.")

Total: 125.0
Category revenue:
 Category
Accessories    40.0
Books          60.0
Stationery     25.0
Name: Revenue, dtype: float64
City revenue (unknown retained):
 City
Berlin    20.0
Leeds     65.0
NaN       40.0
Name: Revenue, dtype: float64
Validation checks passed.


## Quick reference

| Task | Typical syntax |
|---|---|
| Label / position selection | `s.loc["Ahmed"]` / `s.iloc[2]` |
| Filter rows | `df.loc[df["Score"] >= 60]` |
| Rank | `df["Score"].rank(ascending=False)` |
| Count missing | `df.isna().sum()` |
| Fill missing | `s.fillna(s.median())` |
| Stack rows | `pd.concat([jan, feb], ignore_index=True)` |
| Join by column | `orders.merge(customers, on="CustomerID", how="left")` |
| Validate join | `validate="many_to_one"` |
| Audit unmatched | `indicator=True` and filter `left_only` |
| Include missing groups | `groupby("City", dropna=False)` |

**Next:** Work through Activities 1–5. Explain your decisions, not just the code syntax.